# Phase 7 — Persistent Memory & Session Isolation (Bonus)

> Run AFTER Phase 7 (restart kernel first). Cells 1–3 are LLM-free; cell 4 calls Gemini twice.

**The key idea:** `thread_id = f'{client_id}-{session_id}'` is the data-isolation boundary — plus a
ContextVar-based access-control interceptor at the repository chokepoint as belt-and-braces.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env OK')

env OK


## 1. Long-term memory store (per-client, survives restarts)

In [2]:
from app.memory.store import get_memory_store
store = get_memory_store()
for d in store.get_recent_decisions('CLT-001', limit=3):
    print(f"[{d['ts']}] ({d['session_id']}) Q: {d['query'][:60]}")
    print(f"   A: {d['answer'][:100]}...")

[2026-07-13T06:10:34] (demo-2) Q: What did we discuss last time?
   A: On July 13, 2026, you asked what stocks you owned. I informed you that you did not hold any individu...
[2026-07-13T06:10:07] (demo-1) Q: What stocks do I own?
   A: You do not currently hold any individual stocks. Your portfolio consists of the following ETFs and c...


## 2. Access control — cross-client reads die at the repository chokepoint

In [3]:
from app.guardrails.access_control import set_session_client
from app.data.repositories import portfolio_repo

set_session_client('CLT-002')      # simulate an authenticated CLT-002 session
print('own data      :', portfolio_repo.get('CLT-002').client_id, 'OK')
try:
    portfolio_repo.get('CLT-001')  # reach for someone else's book
except PermissionError as e:
    print('cross-client :', 'BLOCKED —', e)
set_session_client(None)           # release for the rest of the notebook

{"rows": 84, "clients": 10, "event": "portfolios_loaded", "level": "info", "timestamp": "2026-07-13T00:48:25.175658Z"}
own data      : CLT-002 OK
{"session_client": "CLT-002", "requested": "CLT-001", "event": "access_denied", "level": "warning", "timestamp": "2026-07-13T00:48:25.178905Z"}
cross-client : BLOCKED — Client CLT-002 may not access data for client CLT-001


## 3. Concurrent sessions can't contaminate each other (ContextVar is task-local)

In [4]:
import asyncio
from app.guardrails.access_control import set_session_client

async def session(own, other):
    set_session_client(own)
    await asyncio.sleep(0.01)
    mine = portfolio_repo.get(own).client_id
    try:
        portfolio_repo.get(other); denied = False
    except PermissionError:
        denied = True
    return f'{own}: reads own ✓, other denied={denied}'

results = await asyncio.gather(session('CLT-001','CLT-002'), session('CLT-002','CLT-001'))
for r in results: print(r)
set_session_client(None)

{"session_client": "CLT-001", "requested": "CLT-002", "event": "access_denied", "level": "warning", "timestamp": "2026-07-13T00:49:24.946856Z"}
{"session_client": "CLT-002", "requested": "CLT-001", "event": "access_denied", "level": "warning", "timestamp": "2026-07-13T00:49:24.949577Z"}
CLT-001: reads own ✓, other denied=True
CLT-002: reads own ✓, other denied=True


## 4. Cross-session recall through the full graph (Gemini)
Session A asks a question; a FRESH session B recalls it via long-term memory.

In [5]:
from langchain_core.messages import HumanMessage
from app.graph.builder import GraphBuilder
from app.agents.base import _text
from langchain_core.messages import AIMessage

graph = GraphBuilder().with_all().build()

def ask(client, session, q):
    cfg = {'configurable': {'thread_id': f'{client}-{session}'}}
    out = graph.invoke({'messages': [HumanMessage(content=q)], 'client_id': client,
                        'session_id': session}, cfg)
    return next((_text(m.content) for m in reversed(out['messages'])
                 if isinstance(m, AIMessage) and m.content), '(no answer produced)')

print('SESSION nb-A:', ask('CLT-001', 'nb-A', 'What is my VTI position worth?')[:150], '\n')
print('SESSION nb-B:', ask('CLT-001', 'nb-B', 'What did we discuss last time?')[:400])

{"client_id": "CLT-001", "prior_decisions": 2, "event": "memory_read", "level": "info", "timestamp": "2026-07-13T00:51:48.299007Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"strategy": "llm", "decision": "portfolio", "event": "route_decision", "session_id": "nb-A", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:51:56.584644Z"}
{"next_agent": "portfolio", "hop": 1, "event": "supervisor_dispatch", "session_id": "nb-A", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:51:56.585594Z"}
{"query": "What is my VTI position worth?", "event": "agent_start", "session_id": "nb-A", "agent": "portfolio", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:51:56.588649Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"method": "get_quote", "source": "finnhub", "event": "chain_served", "session_id": "nb-A", "agent": "portfolio", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:52:05.285888Z"}


...

{"strategy": "llm", "decision": "portfolio", "event": "route_decision", "session_id": "nb-B", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:52:26.790899Z"}
{"next_agent": "portfolio", "hop": 1, "event": "supervisor_dispatch", "session_id": "nb-B", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:52:26.791815Z"}
{"query": "What did we discuss last time?", "event": "agent_start", "session_id": "nb-B", "agent": "portfolio", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:52:26.793937Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"seconds": 7.59, "new_messages": 1, "tool_calls": 0, "event": "agent_done", "session_id": "nb-B", "agent": "portfolio", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:52:34.384204Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"strategy": "llm", "decision": "END", "event": "route_decision", "session_id": "nb-B", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:52:41.914692Z"}
{"hops": 1, "event": "supervisor_end", "session_id": "nb-B", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:52:41.915482Z"}
{"client_id": "CLT-001", "session_id": "nb-B", "event": "decision_saved", "level": "info", "timestamp": "2026-07-13T00:52:41.930549Z"}
SESSION nb-B: On July 13, 2026, you asked about your VTI position and what stocks you owned. I informed you that your VTI position was worth $1,304,415.00 and that you did not hold any individual stocks at that time, with your portfolio consisting of ETFs and cash equivalents.


## ✅ Acceptance check

In [8]:
recall = ask('CLT-001', 'nb-C', 'What did we discuss in previous sessions?').lower()
assert 'vti' in recall or 'discussed' in recall or 'asked' in recall
print('Phase 7 acceptance: PASS — memory recalls, isolation enforced')

AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"strategy": "llm", "decision": "portfolio", "event": "route_decision", "session_id": "nb-C", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:56:13.901034Z"}
{"next_agent": "portfolio", "hop": 1, "event": "supervisor_dispatch", "session_id": "nb-C", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:56:13.901327Z"}
{"query": "What did we discuss in previous sessions?", "event": "agent_start", "session_id": "nb-C", "agent": "portfolio", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:56:13.902390Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"seconds": 15.25, "new_messages": 3, "tool_calls": 1, "event": "agent_done", "session_id": "nb-C", "agent": "portfolio", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:56:29.155090Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"strategy": "llm", "decision": "END", "event": "route_decision", "session_id": "nb-C", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:56:36.320605Z"}
{"hops": 1, "event": "supervisor_end", "session_id": "nb-C", "agent": "supervisor", "client_id": "CLT-001", "level": "info", "timestamp": "2026-07-13T00:56:36.321534Z"}
{"client_id": "CLT-001", "session_id": "nb-C", "event": "decision_saved", "level": "info", "timestamp": "2026-07-13T00:56:36.336792Z"}
Phase 7 acceptance: PASS — memory recalls, isolation enforced
